<style>
table {
  margin-left: 0 !important;
  margin-right: auto !important;
}
th, td {
  text-align: left !important;
}
</style>


## 02-1 · Part 5: Case E, Simulation, and Uncertainty

**One cooling decision can be optimized for one weather case, for an average across cases, or for the worst listed case.**

Parts 1–4 formulated four everyday cases. This part returns to the familiar classroom system from 01-1 and formulates the three weather rules identified in 01-2.

### 1 · First classify function and response structure

The advertising case in Part 2 is linear because its scalar objective and constraints are linear in its continuous decision variables:

> $\displaystyle \underset{x}{\operatorname{minimize}}\quad c^{\mathsf T}x
\qquad\text{subject to}\qquad Ax\le b,\quad Cx=d.$

If an objective or constraint is nonlinear in the optimization variables, the formulation is nonlinear. Squares, products, ratios, \(\max\), and nonlinear response mappings are common sources.

| Case | Function structure | Response evaluation |
|:---|:---|:---|
| B · Advertising | Linear | Direct algebraic |
| D · Study | Nonlinear because \(Q\) contains ratios | Direct algebraic |
| E · Classroom | Nonlinear because \(D\) and \(E\) contain squared terms | Simulation-based |

For the classroom, \(\operatorname{Sim}\) repeats the physical transition \(F\) for \(n=12\) steps and then applies \(G\) to calculate \(D\) and \(E\). The scalar objective \(f(y;\lambda_E)\) is the score supplied by \(H\):

> $\displaystyle f(y;\lambda_E)=J(u;\lambda_E)=D(u)+\lambda_EE(u).$

Simulation-based does not mean uncertain. A simulation with one fixed external-input path is deterministic.

### 2 · Formulate Case E under three weather rules

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/02-1_problem_formulation/assets/case_e_classroom.png" alt="A classroom air conditioner with Early and Late cooling controls and three outdoor weather scenarios of 29, 31, and 33 degrees Celsius." width="570" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

Case E from 01-2: \(x=[u_{\mathrm{early}},u_{\mathrm{late}}]^{\mathsf T}\) is evaluated under the same three weather scenarios. The formulation states how their outcomes are compared.

The decision vector is \(x=[u_{\mathrm{early}},u_{\mathrm{late}}]^{\mathsf T}\in[0,5]^2\). It expands to \(u_0,\ldots,u_{11}\) as in 01-1. For weather scenario \(\xi_s\), the response is

> $\displaystyle y_s=\operatorname{Sim}(x;\xi_s)
=\begin{bmatrix}T_1^{(s)},\ldots,T_{12}^{(s)},D_s(x),E(x)\end{bmatrix}^{\mathsf T}.$

The scenarios use constant outdoor temperatures 29, 31, and 33 °C. Their stated probabilities are \(p=(0.2,0.5,0.3)\).

For scenario \(s\), define \(J_s(x;\lambda_E)=D_s(x)+\lambda_EE(x)\). The three formulations use different rules to combine the scenario scores:

| Rule | Scalar objective |
|:---|:---|
| Deterministic | \(f_{\mathrm{det}}(x)=J_{31}(x;\lambda_E)\) |
| Stochastic | \(f_{\mathrm{sto}}(x)=\sum_{s=1}^{3}p_sJ_s(x;\lambda_E)\) |
| Robust over the listed set | \(f_{\mathrm{rob}}(x)=\max_{s=1,2,3}J_s(x;\lambda_E)\) |

All three formulations use the cooling bounds and energy requirement. The deterministic formulation checks \(20\le T_t\le30\,^\circ\mathrm C\) for the 31 °C path. The stochastic and robust formulations check that safety range in all three listed scenarios. The probability-weighted objective and the worst-case objective still provide different comparison rules.

For the weather cases included by a rule, write the requirements as

> $\displaystyle E(x)-E_{\max}\le0,\qquad
T_{\min}-T_t^{(s)}\le0,\qquad T_t^{(s)}-T_{\max}\le0,$
>
> $\displaystyle t=1,\ldots,12,\qquad
(T_{\min},T_{\max},E_{\max})=(20,30,60).$

The deterministic rule fixes one external-input path. The stochastic rule uses probabilities. The robust rule protects against the worst case in the stated finite uncertainty set; it makes no claim about temperatures outside that set.

### 3 · Evaluate the same candidates under all three rules

The code keeps fixed parameters, external-input scenarios, simulation, performance calculation, feasibility checking, and selection visibly separate. It searches a 0.5-unit cooling grid for a transparent comparison.

In [ ]:
import numpy as np

# Horizon and initial state
TIME_STEPS = 12
INITIAL_TEMPERATURE = 27.0

# Fixed parameters
WEATHER_EXCHANGE = 0.12
OCCUPANT_HEAT = 0.012
COOLING_EFFECT = 0.45

# External-input scenarios
OUTDOOR_SCENARIOS = np.array([29.0, 31.0, 33.0])
SCENARIO_PROBABILITIES = np.array([0.2, 0.5, 0.3])
OCCUPANTS = np.full(TIME_STEPS, 20.0)

# Requirement limits
MIN_COOLING, MAX_COOLING = 0.0, 5.0
MIN_TEMPERATURE, MAX_TEMPERATURE = 20.0, 30.0
MAX_ENERGY = 60.0


def expand_decision(x):
    early_cooling, late_cooling = np.asarray(x, dtype=float)
    return np.r_[np.full(6, early_cooling), np.full(6, late_cooling)]


def simulate_classroom(x, outdoor_temperature):
    cooling = expand_decision(x)
    temperatures = [INITIAL_TEMPERATURE]
    for people, action in zip(OCCUPANTS, cooling):
        current = temperatures[-1]
        temperatures.append(
            current
            + WEATHER_EXCHANGE * (outdoor_temperature - current)
            + OCCUPANT_HEAT * people
            - COOLING_EFFECT * action
        )
    return cooling, np.asarray(temperatures)


def performance_outputs(cooling, temperatures):
    discomfort = np.sum(
        np.maximum(temperatures[1:] - 24.0, 0.0) ** 2
        + np.maximum(22.0 - temperatures[1:], 0.0) ** 2
    )
    energy = 0.5 * np.sum(cooling**2)
    return float(discomfort), float(energy)


def evaluate_weather_rules(x, energy_weight=1.0):
    scenario_records = []
    for outdoor in OUTDOOR_SCENARIOS:
        cooling, temperatures = simulate_classroom(x, outdoor)
        discomfort, energy = performance_outputs(cooling, temperatures)
        scenario_records.append({
            "outdoor": float(outdoor),
            "temperatures": temperatures,
            "discomfort": discomfort,
            "energy": energy,
            "score": discomfort + float(energy_weight) * energy,
        })

    scores = np.array([record["score"] for record in scenario_records])
    energy = scenario_records[0]["energy"]
    common_requirements = (
        np.all(np.asarray(x) >= MIN_COOLING)
        and np.all(np.asarray(x) <= MAX_COOLING)
        and energy <= MAX_ENERGY
    )
    scenario_feasible = np.array([
        record["temperatures"][1:].min() >= MIN_TEMPERATURE
        and record["temperatures"][1:].max() <= MAX_TEMPERATURE
        for record in scenario_records
    ])
    return {
        "x": tuple(map(float, x)),
        "scenarios": scenario_records,
        "objectives": {
            "deterministic": float(scores[1]),
            "stochastic": float(SCENARIO_PROBABILITIES @ scores),
            "robust": float(scores.max()),
        },
        "feasible": {
            "deterministic": bool(common_requirements and scenario_feasible[1]),
            "stochastic": bool(common_requirements and np.all(scenario_feasible)),
            "robust": bool(common_requirements and np.all(scenario_feasible)),
        },
    }

In [ ]:
levels = np.arange(MIN_COOLING, MAX_COOLING + 0.25, 0.5)
records = [
    evaluate_weather_rules([early, late], energy_weight=1.0)
    for early in levels
    for late in levels
]

for rule in ("deterministic", "stochastic", "robust"):
    selected = min(
        (record for record in records if record["feasible"][rule]),
        key=lambda record: record["objectives"][rule],
    )
    print(
        f"{rule:13s}: best grid x={selected['x']}, "
        f"objective={selected['objectives'][rule]:.2f}"
    )

Each line reports the best candidate on the stated 0.5-unit grid under that formulation's objective and weather coverage. The rules can select different candidates because they ask different questions. None of the grid results guarantees a continuous optimum.

### 4 · Classify all five cases on independent axes

| Case | Constraints | Domain | Objectives | Functions | Evaluation | Uncertainty |
|:---|:---|:---|:---|:---|:---|:---|
| A · Clock | Unconstrained | Continuous | Single | Nonlinear | Algebraic | Deterministic |
| B · Advertising | Generally constrained | Continuous | Single | Linear | Algebraic | Deterministic |
| C · Snacks | Generally constrained | Mixed | Single | Linear | Algebraic | Deterministic |
| D · Study | Generally constrained | Continuous | Multiple | Nonlinear | Algebraic | Deterministic |
| E · Classroom, one weather case | Generally constrained | Continuous | Single | Nonlinear | Simulated | Deterministic |
| E · Classroom, probability average | Generally constrained | Continuous | Single | Nonlinear | Simulated | Stochastic |
| E · Classroom, worst listed case | Generally constrained | Continuous | Single | Nonlinear | Simulated | Robust |

These axes are independent. A problem is not merely “continuous” or “nonlinear”; a complete classification combines every applicable label.

The function labels also combine with integer domains. Case C is a mixed-integer linear program (MILP). A continuous linear case is a linear program (LP), a continuous nonlinear case is a nonlinear program (NLP), and a nonlinear case with integer variables is a mixed-integer nonlinear program (MINLP).

### 5 · Separate the formulation from the search algorithm

A formulation states what is chosen, how responses are produced, which candidates are feasible, and how feasible candidates are compared. An algorithm states how the candidate set is searched.

Grid search, gradient-based methods, and evolutionary methods are algorithms. Changing the algorithm while keeping \(x,\mathcal X,\operatorname{Sim},f,g,h\) fixed does not change the optimization problem. Changing any formulation part creates a different problem even if the same algorithm is used.

### Takeaway

Formulate first, then classify along independent axes:

> **identify \(x,\mathcal X,\operatorname{Sim},f,g,h\) → check feasibility before comparison → classify constraints, domain, objectives, functions, evaluation, and uncertainty → choose a suitable search method**

The five cases use the same formulation logic even though their real decisions differ. A change in the decision affects the represented system. A hyperparameter, probability model, uncertainty set, or search grid changes the analysis and must be stated explicitly.